# Legal Document Contradiction Detection using Machine Learning

### Name: Nimit Singh

## Abstract

This project focuses on detecting contradictions in legal documents using Natural Language Processing (NLP) and Machine Learning techniques. The system utilizes the ANLI (Adversarial Natural Language Inference) dataset, which contains challenging and diverse sentence pairs, to classify relationships into entailment, contradiction, and neutral.

Text preprocessing is performed using SpaCy, and feature extraction is carried out using TF-IDF vectorization. Various machine learning models, including Logistic Regression, Support Vector Machine (SVM), Naive Bayes, and Random Forest, are trained and evaluated.

Experimental results show that the Random Forest model outperforms other models in terms of accuracy and robustness. The proposed system can assist in automating contradiction detection in legal documents, thereby improving efficiency and reducing manual effort in legal analysis.

## Problem Statement

Legal documents are complex and lengthy, making it difficult to manually detect contradictions. Identifying such inconsistencies is important to avoid legal disputes.

The objective is to build a machine learning model that classifies relationships between two sentences as:
- Entailment
- Contradiction
- Neutral

## Objectives

- Perform text preprocessing using NLP techniques
- Convert text into numerical form using TF-IDF
- Train multiple machine learning models
- Evaluate and compare model performance
- Identify the best model for contradiction detection

## Dataset

The dataset used in this project is the **ANLI (Adversarial Natural Language Inference)** dataset.

### Features:
- **context:** Original statement (premise)  
- **hypothesis:** Statement to compare  
- **label:** Relationship (entailment, neutral, contradiction)  

### Preprocessing Steps:
- Selected required columns  
- Removed invalid values  
- Combined text data  
- Applied text preprocessing  
- Balanced dataset using SMOTE  

## Importing Required Libraries

In this section, we import all necessary libraries required for data processing, model building, and evaluation.

In [1]:
import pandas as pd 
import spacy
nlp = spacy.load("en_core_web_sm")
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt 
import seaborn as sns 

## Data Loading

The dataset is loaded into a pandas DataFrame.  
We inspect the dataset structure, columns, and sample data.

In [2]:
df = pd.read_json('anli_v1.0/R3/train.jsonl',lines = True)

In [3]:
df.head()

,uid,context,hypothesis,label,model_label,emturk,genre,reason,tag
0,2093cfb3-a15f-4282-81e3-0cb793ffd0d7,"TOKYO, Dec 18 (Reuters) - Japan’s Shionogi & C...",The article was written on December 18th.,e,n,False,news,"TOKYO, Dec 18 (Reuters) is when the article wa...",r3_train
1,d7331d13-1b3b-4423-a3b2-8ac71951b64f,Tallahassee Mayor and Democratic gubernatorial...,Gillum was on TV urging residents to stay out ...,e,c,False,news,This is corrected because he blanketed the nat...,r3_train
2,cc9edb93-10f2-4987-8ae4-165a4fa9806e,MELBOURNE will look to avoid stumbling against...,Carlton beat Melbourne in 2016 and will attemp...,e,n,False,news,"In a game, one team always tries to beat the o...",r3_train
3,6be3c17c-04e4-44a4-a669-92812f003c7e,"by Ted Raymond, Newstalk 580 CFRA A stretch of...",The road was closed for more than two hours af...,e,n,False,news,The road was closed from 12:00 to 6:30. I thin...,r3_train
4,4c126bd1-2fda-4266-81a2-80ae5e3df18d,Drivers are reporting heavy traffic on the nor...,Its advisible to slow down,e,n,False,news,speed has been reduced,r3_train


* Selecting only the required columns (`context`, `hypothesis`, and `label`) from the dataset for further processing.

In [4]:
df = df[['context','hypothesis','label']]

* Displaying the dataset size and class distribution to understand the number of samples and balance of labels.

In [5]:
print("Total Dataset Shape:", df.shape)
print("\nClass Distribution:")
print(df['label'].value_counts())

Total Dataset Shape: (100459, 3)

Class Distribution:
label
n    40778
e    32292
c    27389
Name: count, dtype: int64


* Calculating the frequency of each label to examine how the target classes are distributed in the dataset.

In [6]:
df['label'].value_counts()

label
n    40778
e    32292
c    27389
Name: count, dtype: int64

* Combining the `context` and `hypothesis` columns into a single `Content` column for unified text processing.

In [7]:
df["Content"] = df['context'] + " "  + df['hypothesis']

## Data Preprocessing

- Combined sentence1 and sentence2
- Removed stopwords
- Removed punctuation
- Applied lemmatization using SpaCy

In [8]:
def preprocess(text):
    doc = nlp(text)
    filtered_tokens = []
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        filtered_tokens.append(token.lemma_)
    
    return " ".join(filtered_tokens) 

* Applying the preprocessing function to clean and transform text data, storing the result in `Content_Preprocess`.

In [ ]:
df['Content_Preprocess'] = df['Content'].apply(preprocess)

In [ ]:
df.head()

,context,hypothesis,label,Content,Content_Preprocess
0,"TOKYO, Dec 18 (Reuters) - Japan’s Shionogi & C...",The article was written on December 18th.,e,"TOKYO, Dec 18 (Reuters) - Japan’s Shionogi & C...",TOKYO Dec 18 Reuters Japan Shionogi Co say Tue...
1,Tallahassee Mayor and Democratic gubernatorial...,Gillum was on TV urging residents to stay out ...,e,Tallahassee Mayor and Democratic gubernatorial...,Tallahassee Mayor democratic gubernatorial can...
2,MELBOURNE will look to avoid stumbling against...,Carlton beat Melbourne in 2016 and will attemp...,e,MELBOURNE will look to avoid stumbling against...,MELBOURNE look avoid stumble Carlton late 2016...
3,"by Ted Raymond, Newstalk 580 CFRA A stretch of...",The road was closed for more than two hours af...,e,"by Ted Raymond, Newstalk 580 CFRA A stretch of...",Ted Raymond Newstalk 580 CFRA stretch Highway ...
4,Drivers are reporting heavy traffic on the nor...,Its advisible to slow down,e,Drivers are reporting heavy traffic on the nor...,driver report heavy traffic northbound M6 morn...


 * Mapping categorical labels (n, e, c) to numerical values (0, 1, 2) for model training.

In [ ]:
df['label']=df['label'].map({'n':0,'e':1,'c':2})

Assigning preprocessed text data to `X` (features) and corresponding labels to `y` (target variable) for model training.

In [ ]:
X = df['Content_Preprocess']
y = df['label']

Initializing TF-IDF vectorizer to convert text into numerical features using unigrams and bigrams with feature limits and frequency filtering.

In [ ]:
vectorizer = TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,max_df=0.9)

In [ ]:
X = vectorizer.fit_transform(X)

Importing SMOTE to handle class imbalance by generating synthetic samples for minority classes.

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
smote = SMOTE(sampling_strategy='auto')

* Checking class distribution before applying SMOTE to identify any imbalance in the dataset.

In [ ]:
y.value_counts()

label
0    40778
1    32292
2    27389
Name: count, dtype: int64

Applying SMOTE to balance the dataset by generating synthetic samples, resulting in resampled features `X_sm` and labels `y_sm`.m

In [ ]:
X_sm,y_sm = smote.fit_resample(X,y)

* Verifying class distribution after applying SMOTE to confirm that the dataset is now balanced.

In [ ]:
y_sm.value_counts()

label
1    40778
0    40778
2    40778
Name: count, dtype: int64

## Train-Test Split

- Training Data: 80%
- Testing Data: 20%
- random_state = 42

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_sm, y_sm, test_size=0.20, random_state=42)

## Feature Extraction

TF-IDF Vectorization is used to convert text into numerical features.

- max_features = 15000
- ngram_range = (1,2)

## Models Used

1. Logistic Regression  
2. Support Vector Machine (SVM)  
3. Multinomial Naive Bayes
4. KNN  
5. Random Forest 

# Logistic Regression 

In [ ]:
clf_LR = LogisticRegression(max_iter=2000)

In [ ]:
clf_LR.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [ ]:
y_pred_LR = clf_LR.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(y_test,y_pred_LR))

              precision    recall  f1-score   support

           0       0.62      0.61      0.62      8119
           1       0.62      0.61      0.61      8136
           2       0.62      0.65      0.64      8212

    accuracy                           0.62     24467
   macro avg       0.62      0.62      0.62     24467
weighted avg       0.62      0.62      0.62     24467



# Support Vector Machine (SVM) 

In [ ]:
clf_SVC =LinearSVC()

In [ ]:
clf_SVC.fit(X_train,y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [ ]:
y_pred_SVC = clf_SVC.predict(X_test)

In [ ]:
print(classification_report(y_test,y_pred_SVC))

              precision    recall  f1-score   support

           0       0.66      0.64      0.65      8119
           1       0.64      0.63      0.64      8136
           2       0.65      0.68      0.67      8212

    accuracy                           0.65     24467
   macro avg       0.65      0.65      0.65     24467
weighted avg       0.65      0.65      0.65     24467



# Multinomial Naive Bayes 

In [ ]:
clf_NB = MultinomialNB()

In [ ]:
clf_NB.fit(X_train,y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [ ]:
y_pred_NB = clf_NB.predict(X_test)

In [ ]:
print(classification_report(y_test,y_pred_NB))

              precision    recall  f1-score   support

           0       0.60      0.52      0.56      8119
           1       0.57      0.57      0.57      8136
           2       0.56      0.64      0.60      8212

    accuracy                           0.58     24467
   macro avg       0.58      0.58      0.57     24467
weighted avg       0.58      0.58      0.57     24467



# KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
clf_knn = KNeighborsClassifier()

In [ ]:
clf_knn.fit(X_train,y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [ ]:
y_pred_knn = clf_knn.predict(X_test)

In [ ]:
print(classification_report(y_test,y_pred_knn))

              precision    recall  f1-score   support

           0       0.71      0.64      0.67      8119
           1       0.68      0.70      0.69      8136
           2       0.70      0.74      0.72      8212

    accuracy                           0.70     24467
   macro avg       0.70      0.70      0.70     24467
weighted avg       0.70      0.70      0.70     24467



# Random Forest

In [ ]:
clf_Random = RandomForestClassifier(n_estimators=200,
    max_depth=None,   
    min_samples_split=2,
    min_samples_leaf=1
)

In [ ]:
clf_Random.fit(X_train,y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
y_pred_Random = clf_Random.predict(X_test)

In [ ]:
print(classification_report(y_test,y_pred_Random))

              precision    recall  f1-score   support

           0       0.73      0.71      0.72      8119
           1       0.71      0.72      0.72      8136
           2       0.74      0.75      0.75      8212

    accuracy                           0.73     24467
   macro avg       0.73      0.73      0.73     24467
weighted avg       0.73      0.73      0.73     24467



In [ ]:
import joblib

In [ ]:
joblib.dump(clf_Random,'legal_ANLI_model.joblib')

['legal_ANLI_model.joblib']

In [ ]:
joblib.dump(vectorizer,'legal_ANLI_model_vectorizer.joblib')

['legal_ANLI_model_vectorizer.joblib']

## Conclusion

This project demonstrates how machine learning can be used to detect contradictions in text. Among all models, Random Forest achieved the best performance.

However, accuracy is still moderate, indicating that traditional machine learning models have limitations in understanding complex language.

## Future Work

- Use deep learning models like BERT or RoBERTa
- Improve preprocessing techniques
- Use domain-specific legal datasets
- Hyperparameter tuning
- Build a web application